In [ ]:
# Example 1: Instantiate the URLLibInstrumentor and inspect its dependencies.
from opentelemetry.instrumentation.urllib import URLLibInstrumentor

instrumentor = URLLibInstrumentor()
deps = instrumentor.instrumentation_dependencies()
print(f"instrumentor = {instrumentor!r}")
print(f"instrumentation_dependencies() = {list(deps)}")
assert isinstance(instrumentor, URLLibInstrumentor)
assert deps is not None

In [ ]:
# Example 2: Inspect the package version constant.
from opentelemetry.instrumentation.urllib.version import __version__ as instr_version

print(f"opentelemetry-instrumentation-urllib version: {instr_version}")
assert isinstance(instr_version, str)
assert len(instr_version) > 0

In [ ]:
# Example 3: Instrument urllib, request a local in-process HTTP server, inspect captured spans.
# (A stub/mock opener would replace the instrumented OpenerDirector.open and capture nothing —
# a real request against a loopback server exercises the genuine wrapped path, offline.)
import http.server
import threading
import urllib.request
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.instrumentation.urllib import URLLibInstrumentor

class _Handler(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_response(200); self.send_header("Content-Length", "2"); self.end_headers()
        self.wfile.write(b"ok")
    def log_message(self, *args): pass

server = http.server.HTTPServer(("127.0.0.1", 0), _Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
provider = TracerProvider()
exporter = InMemorySpanExporter()
provider.add_span_processor(SimpleSpanProcessor(exporter))

URLLibInstrumentor().instrument(tracer_provider=provider)
try:
    with urllib.request.urlopen(f"http://127.0.0.1:{server.server_address[1]}/") as resp:
        print(f"status = {resp.status}")
finally:
    URLLibInstrumentor().uninstrument()
    server.shutdown()

spans = exporter.get_finished_spans()
print(f"Captured {len(spans)} span(s): {[s.name for s in spans]}")
assert len(spans) >= 1

In [ ]:
# Example 4: Test the instrument/uninstrument lifecycle and idempotency.
from opentelemetry.instrumentation.urllib import URLLibInstrumentor

instrumentor = URLLibInstrumentor()

# Instrument once, then uninstrument cleanly.
instrumentor.instrument()
assert instrumentor.is_instrumented_by_opentelemetry is True
instrumentor.uninstrument()
assert instrumentor.is_instrumented_by_opentelemetry is False

# Multiple instrument calls should be safe (idempotent at the public API).
instrumentor.instrument()
instrumentor.instrument()
assert instrumentor.is_instrumented_by_opentelemetry is True
instrumentor.uninstrument()
print("instrument/uninstrument lifecycle works")

In [ ]:
# Example 5: instrument() with a custom name_callback hook is accepted.
from opentelemetry.instrumentation.urllib import URLLibInstrumentor

def custom_name_callback(method, url):
    return f"custom-{method}-span"

def custom_request_hook(span, request):
    pass

def custom_response_hook(span, request, response):
    pass

instrumentor = URLLibInstrumentor()
instrumentor.instrument(
    name_callback=custom_name_callback,
    request_hook=custom_request_hook,
    response_hook=custom_response_hook,
)
assert instrumentor.is_instrumented_by_opentelemetry is True
instrumentor.uninstrument()
assert instrumentor.is_instrumented_by_opentelemetry is False
print("Custom hooks accepted by instrument()")